# Abstract Data Types: Starting with the Linked List

**Learning goals for this notebook**

1. Recall the difference between an *Abstract Data Type* (what it does) and a *data structure* (how it's built).
2. Implement a singly linked list from scratch using a `Node` class.
3. Write and test the four basic operations: **get the n-th element**, **search**, **insert**, and **delete**.
4. Compare the cost of these operations to the same operations on a Python `list`.

Run each cell in order (`Shift + Enter`). Cells marked **✏️ Exercise** are for you to complete.

## 1. Quick recap: ADT vs. data structure

An **Abstract Data Type (ADT)** is a *contract*: a description of what operations exist and what they should do, without saying how they're implemented.

For example, the **List ADT** promises operations like:

| Operation | Meaning |
|---|---|
| `get(i)` | return the item at position `i` |
| `search(value)` | find the position of `value` (or report it's missing) |
| `insert(i, value)` | put `value` at position `i`, shifting later items |
| `delete(i)` | remove the item at position `i` |
| `set(i, value)` | replace the item at position `i` |
| `length()` / `is_empty()` | how many items are stored / whether there are none |
| `clear()` | remove everything |

A **data structure** is a concrete way of building that contract. Two very different structures can satisfy the same List ADT:

* **Python's `list`** — a *dynamic array*: one contiguous block of memory holding references, resized (usually doubled) when it fills up. Indexing is instant, but inserting/deleting near the front means shifting everything after it.
* **A linked list** — a chain of small `Node` objects, each pointing to the next. Nothing is contiguous, so there's no shifting… but there's also no instant indexing.

Same interface, different trade-offs. That's the whole point of the ADT idea.

## 2. The `Node` class

A linked list is built from nodes. Each node stores two things:

* `data` — the value we actually care about
* `next` — a reference to the next node in the chain (or `None` if this is the last one)

That's it. A node knows nothing about the list as a whole — only about its neighbour.

In [ ]:
class Node:
    """A single element of a singly linked list."""

    def __init__(self, data, next=None):
        self.data = data
        self.next = next

    def __repr__(self):
        return f"Node({self.data!r})"


# A tiny chain built by hand: 10 -> 20 -> 30
third = Node(30)
second = Node(20, third)
first = Node(10, second)

# Walk the chain by following .next
current = first
while current is not None:
    print(current.data, end=" -> ")
    current = current.next
print("None")

**Check your understanding:** in the cell above, `first.next.next.data` is `30`. What is `first.next.next.next`?

Notice that all we hold onto is `first` — the *head*. Everything else is reachable by following pointers. This is exactly why a linked list only needs to store a reference to its head node.

## 3. The `LinkedList` class

Now we wrap that idea in a class that keeps track of the head and provides the List ADT operations.

Read the code carefully. Every method follows the same basic pattern: **start at the head and walk forward until you find what you need.** You'll see that same `for _ in range(...): current = current.next` walk in `get`, `insert`, and `delete` — that repetition is deliberate, so each method is self-contained and readable on its own.

A few conventions used here:

* Positions are **0-based**, just like Python lists.
* `get` and `delete` raise `IndexError` for a bad position, matching how a Python `list` behaves.
* `search` returns the position of the *first* match, or `-1` if the value isn't there.
* `insert(i, value)` puts the new value **at** position `i`, so `insert(0, x)` prepends and `insert(len, x)` appends.
* `set(i, value)` overwrites the data in an existing node; `clear()` empties the list. Both are tiny, but a List ADT is expected to offer them.

In [ ]:
class LinkedList:
    """A singly linked list implementing the basic List ADT operations."""

    def __init__(self):
        self.head = None
        self._size = 0

    def __len__(self):
        return self._size

    def is_empty(self):
        return self.head is None

    def __repr__(self):
        """Show the list as  head -> a -> b -> c -> None"""
        parts = []
        current = self.head
        while current is not None:
            parts.append(repr(current.data))
            current = current.next
        return "head -> " + " -> ".join(parts + ["None"])

    def clear(self):
        """Remove every element (drop the head and let Python garbage-collect the chain)."""
        self.head = None
        self._size = 0

    # ---------- the four core operations ----------

    def get(self, index):
        """Return the data stored at position `index` (0-based)."""
        if index < 0 or index >= self._size:
            raise IndexError(f"index {index} out of range for list of size {self._size}")

        # Start at the head and take `index` steps forward.
        current = self.head
        for _ in range(index):
            current = current.next
        return current.data

    def set(self, index, value):
        """Replace the data at position `index` with `value` (the node stays put)."""
        if index < 0 or index >= self._size:
            raise IndexError(f"index {index} out of range for list of size {self._size}")

        current = self.head
        for _ in range(index):
            current = current.next
        current.data = value

    def search(self, value):
        """Return the position of the first node holding `value`, or -1 if not found."""
        current = self.head
        position = 0
        while current is not None:
            if current.data == value:
                return position
            current = current.next
            position += 1
        return -1

    def insert(self, index, value):
        """Insert `value` so that it ends up at position `index`.
        Valid indices are 0 .. len(self) inclusive."""
        if index < 0 or index > self._size:
            raise IndexError(f"cannot insert at index {index} in list of size {self._size}")

        new_node = Node(value)

        if index == 0:
            # Special case: the new node becomes the head.
            new_node.next = self.head
            self.head = new_node
        else:
            # General case: walk to the node just BEFORE the insertion point...
            prev = self.head
            for _ in range(index - 1):
                prev = prev.next
            # ...and splice the new node in after it.  (Order matters here!)
            new_node.next = prev.next
            prev.next = new_node

        self._size += 1

    def delete(self, index):
        """Remove the node at position `index` and return its data."""
        if index < 0 or index >= self._size:
            raise IndexError(f"index {index} out of range for list of size {self._size}")

        if index == 0:
            # Special case: removing the head.
            removed = self.head
            self.head = removed.next
        else:
            # General case: walk to the node just BEFORE the one we're removing...
            prev = self.head
            for _ in range(index - 1):
                prev = prev.next
            # ...and route its .next around the victim.
            removed = prev.next
            prev.next = removed.next

        self._size -= 1
        return removed.data

    # ---------- convenience wrappers ----------

    def append(self, value):
        """Add `value` to the end of the list."""
        self.insert(self._size, value)

    def prepend(self, value):
        """Add `value` to the front of the list."""
        self.insert(0, value)

### 3.1 Try it out

In [ ]:
ll = LinkedList()
ll.append("apple")
ll.append("banana")
ll.append("cherry")
print(ll)
print("length:", len(ll))

In [ ]:
# get the n'th element
print("ll.get(0) ->", ll.get(0))
print("ll.get(2) ->", ll.get(2))

In [ ]:
# set replaces data in place; the node (and the list length) don't change
ll.set(1, "blackberry")
print(ll)
ll.set(1, "banana")   # and put it back
print("length still:", len(ll), "| is_empty:", ll.is_empty())

In [ ]:
# search for a value
print("search('banana') ->", ll.search("banana"))
print("search('durian') ->", ll.search("durian"))

In [ ]:
# insert at the front, in the middle, and at the end
ll.insert(0, "avocado")     # front
ll.insert(2, "blueberry")   # middle
ll.insert(len(ll), "date")  # end
print(ll)

In [ ]:
# delete from the front, the middle, and the end
print("removed:", ll.delete(0))
print("removed:", ll.delete(1))
print("removed:", ll.delete(len(ll) - 1))
print(ll)

In [ ]:
# Errors behave like a Python list
try:
    ll.get(99)
except IndexError as e:
    print("IndexError:", e)

### 3.2 Drawing what happens

It's worth sketching these on paper. Here's `insert(2, "X")` on the list `a -> b -> c`:

```
before:   head -> a -> b -> c -> None
                       ^
                      prev (walked 1 step from head)

step 1:   new_node.next = prev.next        # X now points at c
step 2:   prev.next = new_node             # b now points at X

after:    head -> a -> b -> X -> c -> None
```

And `delete(1)` on `a -> b -> c`:

```
before:   head -> a -> b -> c -> None
                  ^    ^
                prev  removed

step:     prev.next = removed.next         # a now points straight at c

after:    head -> a -> c -> None            # b is unreachable and gets garbage-collected
```

**Why does the order of the two assignments in `insert` matter?** Try swapping them in your head and see what breaks.

### 3.3 Interactive playground

Rather than reading about pointer rewiring, try it. Type a value, pick a position with the slider, and click **insert** / **delete** / **get** / **search**. The diagram below updates live; the node that was just touched is highlighted.

Some things to try:
* insert at position 0, then at the last position — notice how the slider's range grows
* delete position 0 — watch the head pointer move
* search for a value that isn't there
* try `get` on a position past the end and see the error

(The code in this cell builds the widget — you don't need to understand it. It only calls the `LinkedList` methods you just read.)

In [ ]:
#@title LinkedList Interactions { display-mode: "form" }
# Double-click the title to see the code (you don't need to understand it).
import ipywidgets as W
from IPython.display import display, HTML

def _render(ll, highlight=None, message=""):
    """Draw the list as boxes and arrows; `highlight` is an index to colour."""
    boxes = ['<div style="display:flex;align-items:center;flex-wrap:wrap;gap:4px;font-family:monospace">'
             '<span style="padding:6px 10px;border-radius:6px;background:#444;color:#fff">head</span>']
    cur, i = ll.head, 0
    while cur is not None:
        bg = "#ffe08a" if i == highlight else "#e8f0fe"
        boxes.append('<span style="margin:0 2px">&#10132;</span>'
                     f'<span style="display:inline-flex;border:2px solid #555;border-radius:6px;overflow:hidden;background:{bg}">'
                     f'<span style="padding:6px 10px;border-right:2px solid #555">{cur.data!r}</span>'
                     f'<span style="padding:6px 8px;color:#888">next</span></span>'
                     f'<span style="font-size:10px;color:#888;margin-left:-48px;margin-top:-34px">[{i}]</span>')
        cur, i = cur.next, i + 1
    boxes.append('<span style="margin:0 2px">&#10132;</span>'
                 '<span style="padding:6px 10px;border-radius:6px;background:#ddd;color:#555">None</span></div>')
    status = f'<div style="margin-top:8px;font-family:sans-serif;color:#333">{message}</div>' if message else ""
    return HTML("".join(boxes) + status + f'<div style="color:#888;font-family:monospace">len = {len(ll)}</div>')

def linked_list_playground(ll=None):
    ll = ll if ll is not None else LinkedList()
    value = W.Text(value="", placeholder="value (e.g. 7 or 'x')", description="value:", layout=W.Layout(width="260px"))
    pos   = W.IntSlider(value=0, min=0, max=len(ll), description="position:", continuous_update=False)
    b_ins = W.Button(description="insert", button_style="success", icon="plus")
    b_del = W.Button(description="delete", button_style="danger", icon="minus")
    b_get = W.Button(description="get", icon="search")
    b_srch= W.Button(description="search", icon="search")
    b_rst = W.Button(description="clear", icon="trash")
    out   = W.Output()

    def parse(text):
        # numbers become ints/floats, everything else stays a string
        try:
            return int(text)
        except ValueError:
            try:
                return float(text)
            except ValueError:
                return text.strip("'\"")

    def refresh(highlight=None, message=""):
        pos.max = len(ll)          # insert allows 0..len, so keep slider in range
        with out:
            out.clear_output(wait=True)
            display(_render(ll, highlight, message))

    def on_insert(_):
        v = parse(value.value)
        try:
            ll.insert(pos.value, v)
            refresh(pos.value, f"insert({pos.value}, {v!r})")
        except IndexError as e:
            refresh(None, f"<span style='color:#c00'>IndexError: {e}</span>")

    def on_delete(_):
        try:
            removed = ll.delete(pos.value)
            refresh(None, f"delete({pos.value}) returned {removed!r}")
        except IndexError as e:
            refresh(None, f"<span style='color:#c00'>IndexError: {e}</span>")

    def on_get(_):
        try:
            refresh(pos.value, f"get({pos.value}) returned {ll.get(pos.value)!r}")
        except IndexError as e:
            refresh(None, f"<span style='color:#c00'>IndexError: {e}</span>")

    def on_search(_):
        v = parse(value.value)
        i = ll.search(v)
        refresh(i if i >= 0 else None, f"search({v!r}) returned {i}" + ("  (not found)" if i < 0 else ""))

    def on_reset(_):
        ll.clear()
        refresh(None, "list cleared")

    b_ins.on_click(on_insert); b_del.on_click(on_delete); b_get.on_click(on_get)
    b_srch.on_click(on_search); b_rst.on_click(on_reset)
    refresh()
    display(W.VBox([W.HBox([value, pos]), W.HBox([b_ins, b_del, b_get, b_srch, b_rst]), out]))

def sorted_playground(sll):
    """Same idea, but for a SortedLinkedList: no position slider, the list decides."""
    value = W.Text(value="", placeholder="value", description="value:", layout=W.Layout(width="260px"))
    b_ins = W.Button(description="insert", button_style="success", icon="plus")
    b_rst = W.Button(description="clear", icon="trash")
    out = W.Output()
    def parse(t):
        try: return int(t)
        except ValueError:
            try: return float(t)
            except ValueError: return t.strip("'\"")
    def refresh(h=None, msg=""):
        with out:
            out.clear_output(wait=True); display(_render(sll, h, msg))
    def on_insert(_):
        v = parse(value.value)
        try:
            sll.insert(v)
            refresh(sll.search(v), f"insert({v!r})")
        except NotImplementedError:
            refresh(None, "<span style='color:#c00'>insert() isn't implemented yet — scroll up and write it!</span>")
        except Exception as e:
            refresh(None, f"<span style='color:#c00'>{type(e).__name__}: {e}</span>")
    def on_reset(_):
        sll.head, sll._size = None, 0; refresh(None, "list cleared")
    b_ins.on_click(on_insert); b_rst.on_click(on_reset); refresh()
    display(W.VBox([W.HBox([value, b_ins, b_rst]), out]))

linked_list_playground()


## 4. Cost comparison: `LinkedList` vs Python `list`

Think about how many nodes each operation has to *touch* in the worst case, for a list of `n` items.

| Operation | `LinkedList` (ours) | Python `list` (dynamic array) |
|---|---|---|
| `get(i)` | walk `i` nodes → **O(n)** | jump straight to slot → **O(1)** |
| `search(v)` | walk until found → **O(n)** | scan until found → **O(n)** |
| `insert(0, v)` | rewire head → **O(1)** | shift everything right → **O(n)** |
| `insert(i, v)` | walk to `i-1`, rewire → **O(n)** | shift items after `i` → **O(n)** |
| `append(v)` | walk to end → **O(n)** *(as written)* | usually free slot → **O(1) amortized** |
| `delete(0)` | rewire head → **O(1)** | shift everything left → **O(n)** |

Two things to notice:

1. Neither structure "wins" — they're good at different things. A linked list shines when you're constantly adding/removing at the **front**; an array shines at **random access**.
2. Our `append` is O(n) only because we walk to the end every time. That's a design choice, not a law — see the exercises.

**A note on `get(i)`.** Positional access is part of the List ADT, so our linked list implements it — but it's the one thing a linked list is *bad* at, and in practice nobody reaches for a linked list when they need `get(i)`. Real linked lists are used for what they're good at: iterating front to back, and adding/removing at the ends (or next to a node you're already holding). Keep that in mind for the next notebook, where we'll build structures that deliberately *drop* `get(i)` from the interface and keep only the operations a linked list does well.

Let's actually measure the front-insert difference:

In [ ]:
import time

N = 20_000

# Python list: insert at front N times
py = []
t0 = time.perf_counter()
for i in range(N):
    py.insert(0, i)
t_list = time.perf_counter() - t0

# Our LinkedList: insert at front N times
ll = LinkedList()
t0 = time.perf_counter()
for i in range(N):
    ll.prepend(i)
t_ll = time.perf_counter() - t0

print(f"Python list  insert(0, x) x{N}: {t_list:.4f} s")
print(f"LinkedList   prepend(x)   x{N}: {t_ll:.4f} s")

Now try the reverse — index into position `N // 2` a few thousand times with each structure — and you should see the tables turn. (That's exercise 1 below.)

## 5. ✏️ Your turn: `SortedLinkedList`

A **sorted linked list** keeps its elements in ascending order at all times. That changes the interface a little: you no longer choose *where* to insert — the list decides based on the value.

So `insert` takes only a `value`, and must walk the list to find the first node whose data is **greater than** `value`, then splice the new node in just before it.

```
list:    head -> 3 -> 7 -> 12 -> None
insert(9):
                       ^ stop here: 12 > 9, so 9 goes between 7 and 12
result:  head -> 3 -> 7 -> 9 -> 12 -> None
```

**Your job:** complete `insert` in the skeleton below. Everything else (`get`, `search`, `delete`, `__repr__`, `__len__`) is inherited from `LinkedList` and already works.

Things to remember from Section 3:
* the special case where the new node becomes the head (empty list, *or* the value is smaller than the current head)
* you need a reference to the node *before* the insertion point
* update `self._size`
* duplicates are allowed — put the new one after existing equal values (or before, your call, but be consistent)

In [ ]:
class SortedLinkedList(LinkedList):
    """A linked list that always keeps its elements in ascending order."""

    def insert(self, value):
        """Insert `value` in the correct sorted position."""
        # ---- YOUR CODE HERE ----
        raise NotImplementedError("insert() not implemented yet")
        # ------------------------

    # These don't make sense on a sorted list, so we disable them.
    def append(self, value):
        raise TypeError("use insert(value) on a SortedLinkedList")

    def prepend(self, value):
        raise TypeError("use insert(value) on a SortedLinkedList")

### 5.1 Quick manual check

Play with your implementation here before running the tests.

The playground below uses **your** `SortedLinkedList` — each insert should land in sorted position automatically.

In [ ]:
s = SortedLinkedList()
for v in [7, 3, 12, 9, 3, 1]:
    s.insert(v)
print(s)          # expected: head -> 1 -> 3 -> 3 -> 7 -> 9 -> 12 -> None
print(len(s))     # expected: 6

In [ ]:
#@title SortedLinkedList Interactions { display-mode: "form" }
sorted_playground(SortedLinkedList())

### 5.2 Run the tests

Just run this cell — you don't need to read or edit `ll_tests.py`. Each line tells you which situation passed (✅), failed (❌), crashed (💥), or hasn't been implemented (⚪). Fix your code and re-run until everything is green.

In [ ]:
#@title Run the tests { display-mode: "form" }
!wget -q https://raw.githubusercontent.com/mdevlin-midpac/ib-cs-2027/main/data_structures/ll_tests.py -O ll_tests.py
from ll_tests import check_sorted_insert
check_sorted_insert(SortedLinkedList)

### 5.3 Stretch: a smarter `search`

Because the list is sorted, `search` can give up early: as soon as it sees a value **larger** than the target, the target can't be further along. Override `search` in `SortedLinkedList` to take advantage of this, then run the second test set.

In [ ]:
!wget -q https://raw.githubusercontent.com/mdevlin-midpac/ib-cs-2027/main/data_structures/ll_tests.py -O ll_tests.py
# Uncomment and complete, or add the method directly to the class above.
#
# def sorted_search(self, value):
#     ...
#
# SortedLinkedList.search = sorted_search

from ll_tests import check_sorted_search
check_sorted_search(SortedLinkedList)

## 6. ✏️ More exercises

Same rules as above — write your solution in the cell provided.

### Exercise 1 — Measure random access

Time `py[N // 2]` vs `ll.get(N // 2)` repeated 2,000 times each. Which one is faster, and roughly by how much? Does this match the table in Section 4?

In [ ]:
# Your code here


### Exercise 2 — Add a `tail` pointer

Right now `append` costs O(n) because we walk to the last node every time. Modify `LinkedList` so it also stores `self.tail`, and update **every** method that could change the last node (`insert`, `delete`, `append`) so `tail` stays correct. Then `append` should be O(1).

Test carefully — the tricky cases are: appending to an empty list, deleting the last node, and deleting the *only* node.

In [ ]:
# Your code here


### Exercise 3 — `__iter__` and `__contains__`

Make the list Pythonic. After this you should be able to write:

```python
for item in ll:
    print(item)

if "banana" in ll:
    ...
```

Hint: `__iter__` can be a generator (`yield`), and `__contains__` can reuse `search`.

In [ ]:
# Your code here


### Exercise 4 — `remove(value)`

Write `remove(value)` that deletes the **first** node containing `value` and returns `True`, or returns `False` if the value isn't present. Try to do it in a single pass (don't call `search` and then `delete` — that walks the list twice).

In [ ]:
# Your code here


### Exercise 5 — `reverse()` (challenge)

Reverse the list **in place** — no new nodes, no extra list. You'll need three variables: `prev`, `current`, and `next_node`. Draw it out first!

```
head -> 1 -> 2 -> 3 -> None      becomes      head -> 3 -> 2 -> 1 -> None
```

In [ ]:
# Your code here


### Exercise 6 — Think about it (no code)

1. Why does `delete` need to find the node **before** the one being removed? What would you need to change about `Node` so you could delete a node given only a reference to *that* node?
2. Python's `list` doubles its capacity when full. Why doubling rather than, say, adding 10 slots each time?
3. Give one real-world situation where you'd pick a linked list over a Python list, and one where you'd do the opposite.

---
## Summary

* An **ADT** describes *what*; a **data structure** describes *how*.
* A **linked list** is a chain of `Node`s, each holding data and a pointer to the next.
* The four core operations — `get`, `search`, `insert`, `delete` — all follow the "start at head, walk forward" pattern; `insert` and `delete` additionally need the node **before** the target.
* Linked lists are O(1) at the front and O(n) for indexing; Python lists are the reverse. Neither is universally better.

Next up: we'll use this same `Node` idea to build **stacks** and **queues** — ADTs with a much more restricted interface, and see why that restriction is a feature, not a bug.